# Hardware-free navigation experiment

This notebook imports the current map and `Navigator` from `Navigation.py`. A fake robot records movement commands so route planning and state updates can be explored without connecting robot hardware.

In [1]:
from Navigation import Navigator, strt, uer6, ler6, uwr6, lwr6

class FakeServo:
    def __init__(self, events):
        self.events = events

    def set_angle(self, angle):
        self.events.append(("servo", angle))

class FakeRobot:
    """Record navigator calls without controlling real hardware."""
    forklift_raise_angle = 200

    def __init__(self, box_polarity=1):
        self.events = []
        self.box_polarity = box_polarity
        self.forklift_servo = FakeServo(self.events)

    def Turn(self, direction):
        self.events.append(("turn", direction))

    def Reverse_Turn(self, direction):
        self.events.append(("reverse_turn", direction))

    def One_Eighty_Degree_Turn(self):
        self.events.append(("turn_180",))

    def Forward_Until_Node(self, speed):
        self.events.append(("forward_to_node", speed))

    def Forward_Until_Rack(self, speed, time=None):
        self.events.append(("forward_to_rack", speed, time))
        return False  # the simulated rack is empty

    def Collect_Box(self):
        self.events.append(("collect_box",))
        return self.box_polarity

    def Drop_Box(self):
        self.events.append(("drop_box",))

print("Loaded current Navigator and map from Navigation.py")

Loaded current Navigator and map from Navigation.py


In [2]:
fake_robot = FakeRobot(box_polarity=1)
navigator = Navigator(robot=fake_robot, strt=strt)

# Select and simulate collection of the nearest box.
box = navigator.get_route_for_box_state(has_box=False)
box_route = list(navigator.route)
print("Target box:", box.name, "(side", box.side, ")")
print("Route to box:", " -> ".join(node.name for node in box_route))
navigator.FollowRoute()

# Current should be the junction beside the box spur, while targeted_box_side
# remembers which side the collected box came from.
print("Current junction after pickup:", navigator.current.name)
print("Remembered box side:", navigator.targeted_box_side)
print("Box polarity:", navigator.polarity)

assert navigator.current is box_route[-2]
assert navigator.targeted_box_side == box.side
assert navigator.polarity == 1
assert ("collect_box",) in fake_robot.events

# Plan the rack route from that junction using the remembered box side.
rack = navigator.get_route_for_box_state(has_box=True)
rack_route = list(navigator.route)
expected_racks = {
    (1, 1): uer6,
    (1, -1): ler6,
    (-1, 1): uwr6,
    (-1, -1): lwr6,
}
assert rack is expected_racks[(box.side, navigator.polarity)]
print("Rack destination:", rack.name)
print("Route to rack:", " -> ".join(node.name for node in rack_route))

# Follow the route, enter the rack spur, and simulate an empty-rack drop-off.
navigator.FollowRoute()
assert navigator.current is rack_route[-2]  # return to the junction by the rack
assert ("forward_to_rack", 80, 2000) in fake_robot.events
assert ("drop_box",) in fake_robot.events
assert fake_robot.events.count(("collect_box",)) == 1
print("Current junction after rack drop:", navigator.current.name)
print("Recorded simulated commands:", fake_robot.events)
print("Navigation simulation passed; no robot hardware was used.")

Target box: leb2 (side 1 )
Route to box: strt -> lej9 -> crse -> lej8 -> leb2
Current junction after pickup: lej8
Remembered box side: 1
Box polarity: 1
Rack destination: uer6
Route to rack: lej8 -> crse -> lej9 -> lcej -> lccj -> ucj6 -> uer6
Current junction after rack drop: ucj6
Recorded simulated commands: [('forward_to_node', 100), ('forward_to_node', 100), ('turn', -1), ('forward_to_node', 100), ('turn', -1), ('collect_box',), ('reverse_turn', -1), ('forward_to_node', 100), ('turn', 1), ('forward_to_node', 100), ('turn', 1), ('forward_to_node', 100), ('turn', -1), ('forward_to_node', 100), ('turn', 1), ('forward_to_node', 100), ('turn', 1), ('forward_to_rack', 80, 2000), ('servo', 200), ('drop_box',), ('reverse_turn', -1)]
Navigation simulation passed; no robot hardware was used.
